# 10 — V1.5 Post-Processing Ablation: Case-Insensitive Executor

Error analysis (notebook 09) showed that a large fraction of v1-fixed errors come from
casing mismatches in WHERE values (e.g., `'Bay Of Islands'` vs `'Bay of Islands'`). The
model identifies the correct column and operator but cannot reproduce the table's exact
casing.

This notebook tests whether a simple executor-side fix — creating text columns with
`TEXT COLLATE NOCASE` so that `=` comparisons ignore letter case — recovers these errors.

**Key point**: the model is NOT re-run. We reuse the predictions already saved in
`results/*.json` and only change the SQLite executor. This is pure post-processing.

**Ablation grid** (2 models × 2 executor modes = 4 cells):

| Model | Case-sensitive executor | Case-insensitive executor |
|---|---|---|
| Base Llama-3-8B | 37.2% (known) | ? |
| V1-fixed (QLoRA + chat prefix) | 51.8% (known) | ? |

In [1]:
import json
import os
import sys
import time

# Put project root on sys.path so `src.*` imports work
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

from src.eval.execution_accuracy import evaluate_dataset

print(f'Project root: {PROJECT_ROOT}')

Project root: /Users/sixuan/Documents/Code/text-to-sql-llama


## 1. Load the test dataset

In [2]:
from datasets import load_dataset

ds = load_dataset('Salesforce/wikisql')
test_examples = list(ds['test'].select(range(500)))
print(f'Loaded {len(test_examples)} test examples')

Loaded 500 test examples


## 2. Load saved predictions

- **Base**: `results/base_model_results.json` (predictions only, no chat prefix, rep_penalty=1.0)
- **V1-fixed**: `results/controlled_comparison_results.json` → `v1_fixed.predictions`

In [3]:
with open(os.path.join(PROJECT_ROOT, 'results', 'base_model_results.json')) as f:
    base_data = json.load(f)
base_predictions = base_data['predictions']
assert len(base_predictions) == 500
print(f'Base predictions loaded: {len(base_predictions)}')
print(f'  Reported accuracy: {base_data["execution_accuracy"]:.1%}')

with open(os.path.join(PROJECT_ROOT, 'results', 'controlled_comparison_results.json')) as f:
    controlled = json.load(f)
v1_fixed_predictions = controlled['v1_fixed']['predictions']
assert len(v1_fixed_predictions) == 500
print(f'V1-fixed predictions loaded: {len(v1_fixed_predictions)}')

Base predictions loaded: 500
  Reported accuracy: 37.2%
V1-fixed predictions loaded: 500


## 3. Sanity check: reproduce the existing numbers

With `case_insensitive=False`, we should recover 37.2% and 51.8% exactly. If not, something
is wrong with the pipeline.

In [4]:
t0 = time.time()
base_original = evaluate_dataset(
    base_predictions, test_examples, case_insensitive=False
)
print(f'Base, case-sensitive:  {base_original["execution_accuracy"]:.1%} '
      f'({base_original["correct"]}/500)  [{time.time() - t0:.1f}s]')

t0 = time.time()
v1_original = evaluate_dataset(
    v1_fixed_predictions, test_examples, case_insensitive=False
)
print(f'V1,   case-sensitive:  {v1_original["execution_accuracy"]:.1%} '
      f'({v1_original["correct"]}/500)  [{time.time() - t0:.1f}s]')

Base, case-sensitive:  37.2% (186/500)  [0.1s]
V1,   case-sensitive:  51.8% (259/500)  [0.0s]


## 4. V1.5 — enable case-insensitive executor

In [5]:
t0 = time.time()
base_ci = evaluate_dataset(
    base_predictions, test_examples, case_insensitive=True
)
print(f'Base, case-insensitive: {base_ci["execution_accuracy"]:.1%} '
      f'({base_ci["correct"]}/500)  [{time.time() - t0:.1f}s]')

t0 = time.time()
v1_ci = evaluate_dataset(
    v1_fixed_predictions, test_examples, case_insensitive=True
)
print(f'V1,   case-insensitive: {v1_ci["execution_accuracy"]:.1%} '
      f'({v1_ci["correct"]}/500)  [{time.time() - t0:.1f}s]')

Base, case-insensitive: 39.6% (198/500)  [0.1s]
V1,   case-insensitive: 68.2% (341/500)  [0.1s]


## 5. Comparison table

In [6]:
print('='*72)
print('V1.5 Post-Processing Ablation — Case-Insensitive Executor')
print('='*72)
print()
print(f'{"Model":20s}  {"Case-sensitive":>18s}  {"Case-insensitive":>18s}  {"Delta":>8s}')
print('-' * 72)

rows = [
    ('Base Llama-3-8B', base_original, base_ci),
    ('V1-fixed (QLoRA)', v1_original, v1_ci),
]
for name, orig, ci in rows:
    orig_acc = orig['execution_accuracy']
    ci_acc = ci['execution_accuracy']
    delta = (ci_acc - orig_acc) * 100
    print(f'{name:20s}  {orig_acc:>17.1%}   {ci_acc:>17.1%}   {delta:>+7.1f}pp')

print()
print('Syntax error rates (should be identical — casing doesn\'t affect parsing):')
for name, orig, ci in rows:
    print(f'  {name:20s}  original={orig["syntax_error_rate"]:.1%}  ci={ci["syntax_error_rate"]:.1%}')

V1.5 Post-Processing Ablation — Case-Insensitive Executor

Model                     Case-sensitive    Case-insensitive     Delta
------------------------------------------------------------------------
Base Llama-3-8B                   37.2%               39.6%      +2.4pp
V1-fixed (QLoRA)                  51.8%               68.2%     +16.4pp

Syntax error rates (should be identical — casing doesn't affect parsing):
  Base Llama-3-8B       original=21.8%  ci=21.8%
  V1-fixed (QLoRA)      original=5.4%  ci=5.4%


## 6. Which examples flipped from wrong → correct?

Inspect v1-fixed predictions that changed status under the case-insensitive executor
to confirm the fix is triggering on casing mismatches (not some other artifact).

In [7]:
from src.eval.execution_accuracy import evaluate_single

flipped = []
for i, (pred, ex) in enumerate(zip(v1_fixed_predictions, test_examples)):
    r_sens = evaluate_single(pred, ex['sql'], ex['table'], case_insensitive=False)
    r_ins = evaluate_single(pred, ex['sql'], ex['table'], case_insensitive=True)
    if (not r_sens['correct']) and r_ins['correct']:
        flipped.append({
            'index': i,
            'question': ex['question'],
            'predicted': pred,
        })

print(f'V1-fixed: {len(flipped)} predictions flipped wrong → correct under case-insensitive.')
print()
print('First 10 examples:')
for ex in flipped[:10]:
    print(f'  [{ex["index"]}] Q: {ex["question"][:70]}')
    print(f'       Pred: {ex["predicted"][:100]}')

V1-fixed: 82 predictions flipped wrong → correct under case-insensitive.

First 10 examples:
  [28] Q: Which province is grey and bell electorate in
       Pred: SELECT `Province` FROM table WHERE `Electorate` = 'grey and bell'
  [29] Q: Which province is bay of islands in
       Pred: SELECT `Province` FROM table WHERE `Electorate` = 'Bay Of Islands'
  [33] Q: what is the maximum ties played where player is josip palada category:
       Pred: SELECT MAX(`Ties played`) FROM table WHERE `Player` = 'Josip Palada Category:Articles With HCards'
  [37] Q: What Branding does WRKO calls use?
       Pred: SELECT `Branding` FROM table WHERE `Calls` = 'wrko'
  [39] Q: Which Market/Rank is associated with WCRN calls?
       Pred: SELECT `Market/Rank` FROM table WHERE `Calls` = 'wcrn'
  [40] Q: Which Frequency is used for WEGP calls?
       Pred: SELECT `Frequency` FROM table WHERE `Calls` = 'wegp'
  [41] Q: What is the regulated retail price for the tariff code ff0 prs?
       Pred: SELECT `BTs r

## 7. Save results

In [8]:
output = {
    'experiment': 'V1.5 post-processing ablation',
    'description': 'Re-evaluate saved predictions with TEXT COLLATE NOCASE executor',
    'n_examples': 500,
    'base_model': {
        'case_sensitive': {
            'execution_accuracy': base_original['execution_accuracy'],
            'syntax_error_rate': base_original['syntax_error_rate'],
            'correct': base_original['correct'],
        },
        'case_insensitive': {
            'execution_accuracy': base_ci['execution_accuracy'],
            'syntax_error_rate': base_ci['syntax_error_rate'],
            'correct': base_ci['correct'],
        },
    },
    'v1_fixed': {
        'case_sensitive': {
            'execution_accuracy': v1_original['execution_accuracy'],
            'syntax_error_rate': v1_original['syntax_error_rate'],
            'correct': v1_original['correct'],
        },
        'case_insensitive': {
            'execution_accuracy': v1_ci['execution_accuracy'],
            'syntax_error_rate': v1_ci['syntax_error_rate'],
            'correct': v1_ci['correct'],
        },
        'flipped_wrong_to_correct': len(flipped),
    },
}

output_path = os.path.join(PROJECT_ROOT, 'results', 'post_processing_ablation.json')
with open(output_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f'Saved: results/post_processing_ablation.json')

Saved: results/post_processing_ablation.json
